# 20 — paper, DOCX edition (FedAgentOps)

Builds `docs/paper/fedagentops_paper.docx`: the FedAgentOps edition of the paper, in the declared
section format. All 13 tables are generated directly from the stored result csvs, all 24
report figures are embedded, and every sentence that quotes a number is an f-string over the
value dictionary recomputed from the same csvs - no figure is typed by hand. The last cell
reopens the saved file, checks the declared per-section word budgets, and pins the headline
numbers, so a changed csv stops the notebook instead of shipping stale prose.

In [1]:
# Goal: load every result csv the document cites and shape the eleven display tables.
# Each table becomes (display strings, bold mask), so the builder cells never touch a number.

from pathlib import Path   # stdlib path objects: keeps the ../data file locations readable
import numpy as np         # numerical library: the argmin/rounding arithmetic behind the bolds
import pandas as pd        # dataframes: every stored result csv is read and pivoted with it

ROOT = Path("..")                     # the notebook executes from src/, the repo root is one up
PROC = ROOT / "data" / "processed"    # result csvs written by the analysis notebooks
EVALS = ROOT / "output" / "evals"     # small evaluation csvs written by the report notebooks
FIGS = ROOT / "output" / "figures"    # the 14 report pngs, embedded unchanged

# Treatment-function names as read from the RTT reference (2026 schema, "Service" suffix
# dropped), the same mapping recorded in fedopd_paper.tex. Codes key every per-specialty csv.
NAMES = {
    "C_100": "General Surgery", "C_101": "Urology", "C_110": "Trauma & Orthopaedics",
    "C_120": "Ear, Nose & Throat", "C_130": "Ophthalmology", "C_140": "Oral Surgery",
    "C_150": "Neurosurgery", "C_160": "Plastic Surgery", "C_170": "Cardiothoracic Surgery",
    "C_300": "General Internal Medicine", "C_301": "Gastroenterology", "C_320": "Cardiology",
    "C_330": "Dermatology", "C_340": "Respiratory Medicine", "C_400": "Neurology",
    "C_410": "Rheumatology", "C_430": "Elderly Medicine", "C_502": "Gynaecology",
    "X01": "Other (pre-2021 schema)", "X02": "Other: Medical", "X03": "Other: Mental Health",
    "X04": "Other: Paediatric", "X05": "Other: Surgical", "X06": "Other: Other",
}


def fmt1(v):
    # every MAE/score cell shares one convention: 1 decimal, thousands comma, "-" if absent
    return "-" if pd.isna(v) else f"{v:,.1f}"


def fmt0(v):
    # national-grain errors are thousands of patients: whole numbers read better there
    return "-" if pd.isna(v) else f"{v:,.0f}"


def bold_mask(num, trained, pers):
    # The bold rule of the three per-specialty tables: mark the single best trained model on
    # the UNROUNDED value where it beats persistence, else mark persistence itself.
    bold = pd.DataFrame(False, index=num.index, columns=num.columns)
    for code, row in num.iterrows():
        if row[trained].isna().all():   # X01 has zero validation rows: nothing to mark
            continue
        best = row[trained].astype(float).idxmin()
        bold.loc[code, best if row[best] < row[pers] else pers] = True
    return bold


TABLES = {}   # table key -> (display DataFrame of strings, bold mask or None)

# ---- Table 1: national era summary ----------------------------------------------------
era = pd.read_csv(EVALS / "eval_national_pre_post.csv")
TABLES["era"] = (pd.DataFrame({
    "era": ["pre-Covid", "post-Covid"],
    "months": era["n_months"].astype(str),
    "mean list": [f"{v:,.0f}" for v in era["mean_waiting"]],
    ">18w share": [f"{v * 100:.1f}%" for v in era["mean_pct_gt18"]],
    ">52w share": [f"{v * 100:.2f}%" for v in era["mean_pct_gt52"]],
}), None)

# ---- Table 2: validation MAE by horizon and grain --------------------------------------
hz = pd.read_csv(PROC / "val_horizons_c110.csv").pivot_table(
    index="h", columns=["model", "regime"], values="mae")
hzc = pd.read_csv(PROC / "val_horizons_coarse_c110.csv").pivot_table(
    index=["grain", "h"], columns=["model", "regime"], values="mae")
H = [1, 3, 6, 12]
HZ_ROWS = [   # grain label, model label, source pivot, (model, regime) column, formatter;
              # challengers shown full-history (post-only is worse everywhere, per the csv)
    ("provider", "persistence", hz, ("persistence", "-"), fmt1),
    ("provider", "drift", hz, ("drift", "full"), fmt1),
    ("provider", "GBM+flows", hz, ("gbm-delta+flows", "full"), fmt1),
    ("provider", "seasonal", hz, ("seasonal", "-"), fmt1),
    ("ICB", "persistence", hzc.loc["icb"], ("persistence", "-"), fmt1),
    ("ICB", "drift", hzc.loc["icb"], ("drift", "full"), fmt1),
    ("national", "persistence", hzc.loc["national"], ("persistence", "-"), fmt0),
    ("national", "drift", hzc.loc["national"], ("drift", "full"), fmt0),
]
TABLES["horizons"] = (pd.DataFrame(
    # .get(h): seasonal stores no h=12 row (the 12-back lag falls outside validation), so
    # that one cell is missing and renders as "-"
    [[g, m] + [f(src[col].get(h)) for h in H] for g, m, src, col, f in HZ_ROWS],
    columns=["grain", "model", "h=1", "h=3", "h=6", "h=12"]), None)

# ---- Table 3: persistence MAE per validation origin month ------------------------------
mo = pd.read_csv(EVALS / "eval_c110_persistence_mae_by_month_h1.csv")
TABLES["monthly"] = (pd.DataFrame({
    "origin month": mo["month"].str[:7],   # "2024-04-01" -> "2024-04"
    "n": mo["n"].astype(str),
    "MAE": [fmt1(v) for v in mo["mae_naive"]],
}), None)

# ---- Table 4: C_110 linear FL summary ---------------------------------------------------
flp = pd.read_csv(PROC / "fl_summary_c110.csv").pivot_table(
    index="model", columns="regime", values=["mae", "rmse"])
FL_ROWS = [   # display name, csv model key, regime key holding its full-history scores;
              # persistence is stored regime-less ("-"), so its post-Covid cell is empty
    ("naive persistence", "naive persistence", "-"),
    ("pooled (centralised)", "pooled", "full"),
    ("FedAvg, delta space", "fedavg-delta", "full"),
    ("FedAvg, delta + local intercept", "fedavg-delta+local-a", "full"),
    ("local-only (level space)", "local-only", "full"),
    ("FedAvg, level space", "fedavg", "full"),
]


def mae_rmse(model, regime):
    # one cell = "MAE / RMSE" for one model in one regime
    v = flp.loc[model, ("mae", regime)]
    return "-" if pd.isna(v) else f"{v:.1f} / {flp.loc[model, ('rmse', regime)]:.1f}"


TABLES["fl"] = (pd.DataFrame({
    "model": [d for d, m, r in FL_ROWS],
    "full history (MAE / RMSE)": [mae_rmse(m, r) for d, m, r in FL_ROWS],
    "post-Covid (MAE / RMSE)": [mae_rmse(m, "post") for d, m, r in FL_ROWS],
}), None)

# ---- Table 5: FL scale-up per specialty (linear delta families) -------------------------
sc = pd.read_csv(PROC / "fl_scaleup_by_specialty.csv")   # csv is sorted by n_val descending
SC_COLS = {"persistence": "persistence_mae", "local": "local_full_mae",
           "pooled": "pooled_full_mae", "FedAvg-Δ": "fedavg_delta_full_mae",
           "FedAvg-Δ+intercept": "fedavg_delta_local_a_full_mae"}
sc_num = sc.set_index("tfunc_code")[list(SC_COLS.values())]
sc_num.columns = list(SC_COLS)
sc_disp = pd.DataFrame(index=sc_num.index)
sc_disp["code"] = sc_num.index
sc_disp["treatment function"] = [NAMES[c] for c in sc_num.index]
sc_disp["n_val"] = [f"{v:,}" for v in sc.set_index("tfunc_code")["n_val"]]
sc_disp["clients"] = sc.set_index("tfunc_code")["n_clients"].astype(str)
for c in SC_COLS:
    sc_disp[c] = [fmt1(v) for v in sc_num[c]]
TABLES["scaleup"] = (sc_disp, bold_mask(sc_num, [c for c in SC_COLS if c != "persistence"],
                                        "persistence"))

# ---- Table 6: neural vs linear per specialty, full regime -------------------------------
nn = pd.read_csv(PROC / "fl_neural_summary.csv")
pers_nn = nn[nn["model"] == "persistence"].set_index("tfunc_code")
fullp = nn[nn["regime"] == "full"].pivot(index="tfunc_code", columns="model", values="mae")
NN_COLS = {"linear pooled": "linear_pooled", "linear FedAvg": "linear_fedavg",
           "neural pooled": "neural_pooled", "neural FedAvg": "neural_fedavg"}
nn_num = fullp[list(NN_COLS.values())].copy()
nn_num.columns = list(NN_COLS)
nn_num.insert(0, "persistence", pers_nn["mae"])
nn_disp = pd.DataFrame(index=nn_num.index)
nn_disp["code"] = nn_num.index
nn_disp["treatment function"] = [NAMES[c] for c in nn_num.index]
nn_disp["n"] = [f"{v:,}" for v in pers_nn.loc[nn_num.index, "n"]]
for c in nn_num.columns:
    nn_disp[c] = [fmt1(v) for v in nn_num[c]]
TABLES["neural"] = (nn_disp, bold_mask(nn_num, list(NN_COLS), "persistence"))

# ---- Table 7: declared multi-seed check at C_110 ----------------------------------------
ms = pd.read_csv(PROC / "fl_neural_c110_multiseed.csv")
msp = ms.pivot_table(index=["regime", "model"], columns="seed", values="mae")
ms_rows = []
for reg, mod in [("full", "neural_pooled"), ("full", "neural_fedavg"),
                 ("post", "neural_pooled"), ("post", "neural_fedavg")]:
    v = msp.loc[(reg, mod)]
    ms_rows.append([reg, mod.replace("_", " ")] + [f"{v[s]:.1f}" for s in (42, 123, 999)]
                   + [f"{v.mean():.1f} ± {v.std(ddof=1):.1f}"])   # 3 seeds = a sample: ddof=1
TABLES["multiseed"] = (pd.DataFrame(
    ms_rows, columns=["regime", "model", "seed 42", "seed 123", "seed 999", "mean ± sd"]), None)

# ---- Table 8: frozen agent rules --------------------------------------------------------
ag = pd.read_csv(PROC / "agent_frozen_summary.csv")
RULE_NAMES = {"v1": "capacity", "v2": "redistribution"}   # rule sets named by what they decide;
# the csv keeps the short keys, so an unexpected key in a regenerated csv trips the lookup
TABLES["agent"] = (pd.DataFrame({
    "rule set": [RULE_NAMES[r] for r in ag["rule"]], "decision": ag["decision"],
    "n": [f"{v:,}" for v in ag["n"]],
    "share of 2,337": [f"{v * 100:.1f}%" for v in ag["share_of_2337"]],
    "median pressure": [f"{v:.3f}" for v in ag["median_pressure"]],
    "median 6m change": [f"{v:+,.1f}" for v in ag["median_realized_change"]],
}), None)

# ---- Table 9: v2 stress test ------------------------------------------------------------
st = pd.read_csv(EVALS / "eval_c110_agent_stress_h6.csv")
TABLES["stress"] = (pd.DataFrame({
    "window": st["window"], "class": st["decision_v2"],
    "n": st["n"].astype(str),
    "median pressure": [f"{v:.3f}" for v in st["median_pressure"]],
    "median 6m change": [f"{v:+,.1f}" for v in st["median_realized_change"]],
}), None)

# ---- Table 10: sealed year at the anchor ------------------------------------------------
se = pd.read_csv(PROC / "sealed_year_eval.csv")
c110 = se[se["tfunc_code"] == "C_110"].set_index(["model", "train_cut"])
SE_ROWS = [
    ("persistence", "-", "persistence"),
    ("neural_pooled", "final", "neural pooled (final, seed 42)"),
    ("neural_fedavg", "final", "neural FedAvg (final, seed 42)"),
    ("linear_pooled", "final", "linear pooled (final)"),
    ("linear_fedavg", "final", "linear FedAvg (final)"),
    ("linear_pooled", "val_frozen", "linear pooled (val-frozen)"),
    ("linear_fedavg", "val_frozen", "linear FedAvg (val-frozen)"),
]
TABLES["sealed_anchor"] = (pd.DataFrame({
    "model": [d for m, c, d in SE_ROWS],
    "sealed MAE": [fmt1(c110.loc[(m, c), "mae"]) for m, c, d in SE_ROWS],
    "sealed RMSE": [fmt1(c110.loc[(m, c), "rmse"]) for m, c, d in SE_ROWS],
    "sealed R2 (delta)": [f"{c110.loc[(m, c), 'r2_delta_secondary']:+.3f}" for m, c, d in SE_ROWS],
    "validation MAE": [fmt1(c110.loc[(m, c), "val_mae_full"]) for m, c, d in SE_ROWS],
}), None)

# ---- Table 13: the full metric set at the anchor on validation --------------------------
# MAE alone hides two things the discussion needs: whether a model wins by trimming large
# errors (RMSE) and whether it explains any of the month-to-month change at all (R2 on deltas,
# secondary by declaration - level-R2 stays rejected because provider size dominates it).
MT_ROWS = [("persistence", "persistence"), ("linear_pooled", "linear pooled"),
           ("linear_fedavg", "linear FedAvg"), ("neural_pooled", "neural pooled"),
           ("neural_fedavg", "neural FedAvg")]
c110_nn = nn[(nn["tfunc_code"] == "C_110") & (nn["regime"].isin(["full", "-"]))].set_index("model")
assert len(c110_nn) == 5, c110_nn.index.tolist()   # one row per model; a second regime row trips this
TABLES["metrics"] = (pd.DataFrame({
    "model": [d for m, d in MT_ROWS],
    "MAE": [fmt1(c110_nn.loc[m, "mae"]) for m, d in MT_ROWS],
    "RMSE": [fmt1(c110_nn.loc[m, "rmse"]) for m, d in MT_ROWS],
    "R2 on deltas": [f"{c110_nn.loc[m, 'r2_delta_secondary']:+.3f}" for m, d in MT_ROWS],
    "RMSE / MAE": [f"{c110_nn.loc[m, 'rmse'] / c110_nn.loc[m, 'mae']:.2f}" for m, d in MT_ROWS],
}), None)

# ---- Table 11: sealed year, complete one-shot per specialty -----------------------------
fin = se[(se["train_cut"] != "val_frozen") & (se["model"] != "persistence")].pivot(
    index="tfunc_code", columns="model", values="mae")
vf = se[se["train_cut"] == "val_frozen"].pivot(index="tfunc_code", columns="model", values="mae")
sealedp = se[se["model"] == "persistence"].set_index("tfunc_code")
sl_num = pd.DataFrame({
    "persistence": sealedp["mae"],
    "linear FedAvg": fin["linear_fedavg"], "linear pooled": fin["linear_pooled"],
    "neural FedAvg": fin["neural_fedavg"], "neural pooled": fin["neural_pooled"],
    "val-frozen linear FedAvg": vf["linear_fedavg"],
})
sl_disp = pd.DataFrame(index=sl_num.index)
sl_disp["code"] = sl_num.index
sl_disp["treatment function"] = [NAMES[c] for c in sl_num.index]
sl_disp["n"] = [f"{v:,}" for v in sealedp.loc[sl_num.index, "n"]]
for c in sl_num.columns:
    sl_disp[c] = [fmt1(v) for v in sl_num[c]]
TABLES["sealed_full"] = (sl_disp, bold_mask(
    sl_num, [c for c in sl_num.columns if c != "persistence"], "persistence"))

# ---- Table 12: cohort-rule sensitivity (2026-08-01 audit, notebook 21) ------------------
cs = pd.read_csv(PROC / "cohort_sensitivity.csv")
assert cs.shape == (12, 3), cs.shape


def fmtcs(v):
    # count rows are whole numbers stored as floats; MAE rows keep their two decimals
    return f"{int(v):,}" if v == int(v) else f"{v:.2f}"


TABLES["cohort"] = (pd.DataFrame({
    "quantity": cs["quantity"],
    "stored rule": [fmtcs(v) for v in cs["stored_rule"]],
    "prospective rule": [fmtcs(v) for v in cs["prospective_rule"]],
}), None)

# ---- Pins: headline numbers the prose quotes, recomputed here from the csvs -------------
# ---- V: every number the prose quotes, recomputed here from the same csvs ----------------
# The prose in cell 2 is written as f-strings over this dictionary, so a sentence cannot drift
# from the result it describes: regenerate a csv and the sentence changes with it. Values are
# stored pre-formatted as strings, because the formatting convention (decimals, thousands
# separators, signs) is part of what the sentence claims.

# the four result files the tables above did not already need
pc = pd.read_csv(PROC / "fl_per_client_c110.csv")            # per-client scores at the anchor
pmae = pd.read_csv(EVALS / "eval_c110_provider_mae_h1.csv")  # the distribution behind Fig. 9
bands = pd.read_csv(PROC / "val_conformal_bands_c110.csv").set_index("h")
dec = pd.read_csv(PROC / "agent_decisions_h6_c110.csv", parse_dates=["month"])
csx = cs.set_index("quantity")
fl_n = pd.read_csv(PROC / "fl_summary_c110.csv")["n"].iloc[0]
# a within-ICB percentile rank is 1.0 by construction when a provider is the only one reporting
# that month, so those sender flags are mechanical and are counted apart from the peer-based ones
lone = dec.groupby(["month", "icb_code"])["provider_code"].transform("size") == 1
out = dec["decision_v2"] == "REDISTRIBUTE_OUT"
drift_abs = ((sealedp["mae"] / sealedp["val_mae_full"] - 1) * 100).abs().median()
all_trained = pd.concat([fin, vf.add_prefix("vf_")], axis=1)
val_trained = nn[nn["model"] != "persistence"].pivot_table(
    index="tfunc_code", columns=["model", "regime"], values="mae")
# "measured anywhere" spans both years, so both are searched for the largest relative margin
margin_max = max(((1 - all_trained.div(sealedp["mae"], axis=0)) * 100).max().max(),
                 ((1 - val_trained.div(pers_nn["mae"], axis=0)) * 100).max().max())

V = {}

# national era and panel shape
pre, post = era.iloc[0], era.iloc[1]
V["list_pre"] = f"{pre.mean_waiting / 1e6:.2f} million"
V["list_post"] = f"{post.mean_waiting / 1e6:.2f} million"
V["gt52_pre"] = f"{pre.mean_pct_gt52 * 100:.2f}%"
V["gt52_post"] = f"{post.mean_pct_gt52 * 100:.2f}%"
V["gt52_fold"] = f"{post.mean_pct_gt52 / pre.mean_pct_gt52:.0f}-fold"
V["n_pre"], V["n_post"] = str(int(pre.n_months)), str(int(post.n_months))
V["n_months"] = str(int(era["n_months"].sum()))

panel = pd.read_csv(PROC / "rtt_panel_with_provider_flags.csv")
V["panel_rows"] = f"{len(panel):,}"
V["panel_cols"] = str(pd.read_csv(PROC / "rtt_kh03_canonical_panel.csv", nrows=1).shape[1])
V["providers"] = f"{panel['provider_code'].nunique():,}"
V["n_tfunc"] = str(panel["tfunc_code"].nunique())
pub = panel[panel["is_public"]]
V["public_n"] = str(pub["provider_code"].nunique())
V["public_share"] = f"{pub['total_waiting'].sum() / panel['total_waiting'].sum() * 100:.1f}%"
V["fl_public"] = str(panel[panel["fl_client"] & panel["is_public"]]["provider_code"].nunique())
V["fl_indep"] = str(panel[panel["fl_client"] & ~panel["is_public"]]["provider_code"].nunique())

# the two training samples: full history (panel start to the development cut) and post-Covid
# only (the March 2020 break to the same cut) - the sizes the methodology quotes
mall = pd.to_datetime(panel["month"].drop_duplicates())
V["full_months"] = str(int((mall < "2025-04-01").sum()))
V["post_months"] = str(int(((mall >= "2020-03-01") & (mall < "2025-04-01")).sum()))
V["full_years"] = "seven"   # 84 months; the assert below keeps the word and the number honest
V["post_years"] = "five"    # 61 months
assert (V["full_months"], V["post_months"]) == ("84", "61"), (V["full_months"], V["post_months"])

kh = pd.read_csv(PROC / "kh03_monthly_panel.csv", parse_dates=["month"])
khn = kh.groupby("month")[["avail_ga", "occ_ga"]].sum()
khocc = khn["occ_ga"] / khn["avail_ga"]   # national rate: beds occupied over beds open that month
V["kh_start"] = f"{kh['month'].min():%B %Y}"
V["occ_lo"], V["occ_hi"] = f"{khocc.min() * 100:.1f}%", f"{khocc.max() * 100:.1f}%"

# conformal bands
V["band_h1"] = f"±{bands.loc[1, 'half_width']:,.0f} patients"
V["band_h12"] = f"±{bands.loc[12, 'half_width']:,.0f}"
V["cov_range"] = f"{bands['coverage'].min() * 100:.1f}-{bands['coverage'].max() * 100:.1f}%"

# linear federation at the anchor
V["pers_mae"] = f"{flp.loc['naive persistence', ('mae', '-')]:.1f}"
V["pers_rmse"] = f"{flp.loc['naive persistence', ('rmse', '-')]:.1f}"
V["lvl_full"] = f"{flp.loc['fedavg', ('mae', 'full')]:.1f}"
V["lvl_post"] = f"{flp.loc['fedavg', ('mae', 'post')]:.1f}"
V["dlt_full"] = f"{flp.loc['fedavg-delta', ('mae', 'full')]:.1f}"
V["pool_full"] = f"{flp.loc['pooled', ('mae', 'full')]:.1f}"
V["fed_pool_gap"] = f"{flp.loc['fedavg-delta', ('mae', 'full')] - flp.loc['pooled', ('mae', 'full')]:.1f}"
V["n_anchor"] = f"{int(fl_n):,}"

# per-client picture at the anchor
V["n_clients"] = str(len(pc))
V["fed_beats_local"] = f"{(pc['mae_fedavg'] < pc['mae_local']).mean() * 100:.1f}%"
V["med_local"] = f"{pc['mae_local'].median():.0f}"
V["med_fed"] = f"{pc['mae_fedavg'].median():.0f}"
V["med_naive"] = f"{pc['mae_naive'].median():.0f}"
V["n_prov_fig"] = str(len(pmae))
V["med_prov"] = f"{pmae['mae_naive'].median():.0f}"
V["mo_lo"], V["mo_hi"] = f"{mo['mae_naive'].min():.1f}", f"{mo['mae_naive'].max():.1f}"
V["hz_pers_h1"] = f"{hz[('persistence', '-')].loc[1]:.1f}"   # persistence on the horizon cohort

# scale-up across specialties: every variant against the baseline, and against pooling
scv = sc[sc["n_val"] > 0].set_index("tfunc_code")
V["n_spec"] = str(len(scv))
V["n_val_all"] = f"{sc['n_val'].sum():,}"
fpg = scv["fedavg_delta_full_mae"] - scv["pooled_full_mae"]
V["fp_med"], V["fp_worst"], V["fp_best"] = f"{fpg.median():+.2f}", f"{fpg.max():+.2f}", f"{fpg.min():+.2f}"
VAR = {"local": "local_full_mae", "pool": "pooled_full_mae",
       "dlt": "fedavg_delta_full_mae", "dlta": "fedavg_delta_local_a_full_mae"}
for k, col in VAR.items():
    d = scv[col] - scv["persistence_mae"]
    V[f"{k}_wins"] = f"{int((d < 0).sum())} of {len(d)}"
    V[f"{k}_medgap"] = f"{d.median():+.2f}"
d_dlt = scv["fedavg_delta_full_mae"] - scv["persistence_mae"]
V["any_wins"] = str(int((scv[list(VAR.values())].min(axis=1) - scv["persistence_mae"] < 0).sum()))
bw = d_dlt.idxmin()
V["dlt_best_code"], V["dlt_best_name"] = bw, NAMES[bw]
V["dlt_best_mae"] = f"{scv.loc[bw, 'fedavg_delta_full_mae']:.1f}"
V["dlt_best_pers"] = f"{scv.loc[bw, 'persistence_mae']:.1f}"
V["dlt_worst_loss"] = f"{d_dlt.max():.1f}"          # the biggest loss to persistence, not a win
V["dlt_best_win"] = f"{-d_dlt.min():.1f}"
# regime: post-only minus full history, per linear variant (positive = dropping history hurts)
for k, col in VAR.items():
    r = scv[col.replace("_full_", "_post_")] - scv[col]
    clin = [c for c in r.index if c.startswith("C_")]   # X-codes have no pre-Covid rows to drop
    V[f"{k}_reg_wins"] = f"{int((r > 0).sum())} of {len(r)}"
    V[f"{k}_reg_med"] = f"{r.median():+.2f}"
    V[f"{k}_reg_clin"] = f"{int((r.loc[clin] > 0).sum())} of {len(clin)}"
    V[f"{k}_reg_worst"] = r.idxmax()
    V[f"{k}_reg_worstv"] = f"{r.max():+.1f}"
V["n_xcodes"] = str(len([c for c in scv.index if not c.startswith("C_")]))

# neural against linear, and both against the baseline
V["n_spec_nn"] = str(nn["tfunc_code"].nunique())
for reg in ("full", "post"):
    p = nn[nn["regime"] == reg].pivot(index="tfunc_code", columns="model", values="mae")
    for arch in ("pooled", "fedavg"):
        g = p[f"neural_{arch}"] - p[f"linear_{arch}"]
        V[f"nvl_{arch}_{reg}"] = f"{int((g < 0).sum())} of {len(g)}"
        V[f"nvl_{arch}_{reg}_med"] = f"{g.median():+.2f}"
    if reg == "full":
        # the size question, scaled exactly as notebook 13 scales it for Fig. 16
        rel = (p["neural_fedavg"] - p["linear_fedavg"]) / pers_nn["mae"].reindex(p.index) * 100
        V["r_size"] = f"{np.corrcoef(pers_nn['n'].reindex(p.index), rel)[0, 1]:.3f}"
        V["nn_best_codes"] = ", ".join(rel.nsmallest(4).index)
        nreg = nn[nn["model"] == "neural_fedavg"].pivot(index="tfunc_code", columns="regime", values="mae")
        nr = (nreg["post"] - nreg["full"])
        nclin = [c for c in nr.index if c.startswith("C_")]
        V["neu_reg_clin"] = f"{int((nr.loc[nclin] > 0).sum())} of {len(nclin)}"
        V["neu_reg_worstv"] = f"{nr.max():+.1f}"
V["n_scores"] = str(len(nn[nn["model"] != "persistence"]))   # trained model-regime-specialty scores
V["r2_max"] = f"{nn['r2_delta_secondary'].max():+.3f}"
V["r2_max_code"] = nn.loc[nn["r2_delta_secondary"].idxmax(), "tfunc_code"]
npc = pd.read_csv(PROC / "fl_neural_per_client.csv")
V["nn_pairs"] = f"{len(npc):,}"
V["nn_pair_share"] = f"{(npc['mae_neural_fedavg'] < npc['mae_linear_fedavg']).mean() * 100:.1f}%"
for m, k in [("persistence", "pers"), ("linear_pooled", "lp"), ("linear_fedavg", "lf"),
             ("neural_pooled", "np"), ("neural_fedavg", "nf")]:
    V[f"a_{k}_mae"] = f"{c110_nn.loc[m, 'mae']:.1f}"
    V[f"a_{k}_rmse"] = f"{c110_nn.loc[m, 'rmse']:.1f}"
    V[f"a_{k}_r2"] = f"{c110_nn.loc[m, 'r2_delta_secondary']:+.3f}"
V["n_anchor_nn"] = f"{int(pers_nn.loc['C_110', 'n']):,}"
V["rmse_mae_ratio"] = f"{(c110_nn['rmse'] / c110_nn['mae']).mean():.2f}"
# the within-specialty illustration: where pooling and federating a GRU pull apart
ill = nn[(nn["tfunc_code"] == V["r2_max_code"]) & (nn["regime"].isin(["full", "-"]))].set_index("model")
V["ill_code"], V["ill_name"] = V["r2_max_code"], NAMES[V["r2_max_code"]]
for m, k in [("neural_pooled", "np"), ("neural_fedavg", "nf"), ("linear_fedavg", "lf"),
             ("persistence", "pers")]:
    V[f"ill_{k}"] = f"{ill.loc[m, 'mae']:.1f}"

# agent layer
V["n_dec"] = f"{len(dec):,}"   # each rule set scores the same provider-months
V["dec_providers"] = str(dec["provider_code"].nunique())
V["dec_from"], V["dec_to"] = f"{dec['month'].min():%B %Y}", f"{dec['month'].max():%B %Y}"
V["n_send"] = str(int(out.sum()))
V["n_recv"] = str(int((dec["decision_v2"] == "REDISTRIBUTE_IN").sum()))
V["send_peer"] = f"{int((out & ~lone).sum())} are peer-based"
V["send_lone"] = f"{int((out & lone).sum())} arise in lone-provider"
V["p_send"] = f"{ag.loc[ag['decision'] == 'REDISTRIBUTE_OUT', 'median_pressure'].iloc[0]:.3f}"
V["p_recv"] = f"{ag.loc[ag['decision'] == 'REDISTRIBUTE_IN', 'median_pressure'].iloc[0]:.3f}"
V["p_peer"] = f"{dec.loc[out & ~lone, 'pressure_score'].median():.3f}"
V["p_lonef"] = f"{dec.loc[out & lone, 'pressure_score'].median():.3f}"
V["p_cohort"] = f"{dec['pressure_score'].median():.2f}"
V["send_change"] = f"{ag.loc[ag['decision'] == 'REDISTRIBUTE_OUT', 'median_realized_change'].iloc[0]:+,.0f}"
V["send_change_last"] = f"{st[st['decision_v2'] == 'REDISTRIBUTE_OUT']['median_realized_change'].iloc[-1]:+,.0f}"
V["sq_change_first"] = f"{st[st['decision_v2'] == 'STATUS_QUO']['median_realized_change'].iloc[0]:+,.0f}"
V["exp_change"] = f"{ag.loc[ag['decision'] == 'EXPAND_CAPACITY', 'median_realized_change'].iloc[0]:+,.1f}"
V["sq_change"] = f"{ag.loc[(ag['rule'] == 'v1') & (ag['decision'] == 'STATUS_QUO'), 'median_realized_change'].iloc[0]:+,.0f}"
recv_idx = dec[dec["decision_v2"] == "REDISTRIBUTE_IN"].set_index(["month", "icb_code"]).index
V["has_recv"] = f"{dec[out].set_index(['month', 'icb_code']).index.isin(recv_idx).mean() * 100:.1f}%"
peers = dec.groupby(["month", "icb_code"])["provider_code"].transform("size")
V["small_icb_n"] = f"{int((peers[out] < 4).sum())}"
V["small_icb"] = f"{(peers[out] < 4).mean() * 100:.1f}%"

# sealed year
V["n_sealed"] = f"{int(sealedp.loc['C_110', 'n']):,}"
for m, k in [("neural_pooled", "np"), ("neural_fedavg", "nf"),
             ("linear_pooled", "lp"), ("linear_fedavg", "lf")]:
    g = fin[m] - sealedp["mae"].reindex(fin.index)
    V[f"s_{k}_wins"] = f"{int((g < 0).sum())} of {len(g)}"
    V[f"s_{k}_mae"] = f"{fin.loc['C_110', m]:.1f}"
V["s_vf_wins"] = f"{int((vf['linear_fedavg'] - sealedp['mae'].reindex(vf.index) < 0).sum())} of {len(vf)}"
V["s_pers"] = f"{sealedp.loc['C_110', 'mae']:.1f}"
V["s_pers_r2"] = f"{sealedp.loc['C_110', 'r2_delta_secondary']:+.3f}"
V["s_np_r2"] = f"{se.set_index(['tfunc_code', 'model', 'train_cut']).loc[('C_110', 'neural_pooled', 'final'), 'r2_delta_secondary']:+.3f}"
V["s_anchor_gap"] = f"{fin.loc['C_110', 'neural_pooled'] - sealedp.loc['C_110', 'mae']:.2f}"
sbest = (fin["neural_pooled"] - sealedp["mae"].reindex(fin.index)).idxmin()
V["s_best_code"], V["s_best_name"] = sbest, NAMES[sbest]
V["s_best_mae"] = f"{fin.loc[sbest, 'neural_pooled']:.1f}"
V["s_best_pers"] = f"{sealedp.loc[sbest, 'mae']:.1f}"
V["drift_med"] = f"{((sealedp['mae'] / sealedp['val_mae_full'] - 1) * 100).median():+.1f}%"
V["drift_abs"] = f"absolute change is {drift_abs:.1f}%"
V["drift_lo"] = f"{((sealedp['mae'] / sealedp['val_mae_full'] - 1) * 100).min():+.1f}%"
V["drift_hi"] = f"{((sealedp['mae'] / sealedp['val_mae_full'] - 1) * 100).max():+.1f}%"
V["margin_max"] = f"is {margin_max:.1f}%"
V["s_rmse_anchor"] = f"{sealedp.loc['C_110', 'rmse']:.0f}"
V["s_rmse_mean"] = f"{sealedp['rmse'].mean():.0f}"
V["v_rmse_mean"] = f"{pers_nn['rmse'].mean():.0f}"
V["cs_val"] = f"{csx.loc['validation persistence MAE', 'prospective_rule']:.2f}"
V["cs_sealed"] = f"{csx.loc['sealed persistence MAE', 'prospective_rule']:.2f}"
V["cs_sealed_stored"] = f"{csx.loc['sealed persistence MAE', 'stored_rule']:.2f}"

# Pins: the subset of V the verification cell must find verbatim in the built document. Every
# other V entry is checked implicitly, since the prose is rendered from the same strings.
PINS = [V[k] for k in ("pers_mae", "lvl_full", "dlt_full", "pool_full", "a_np_mae", "a_nf_mae",
                       "a_pers_mae", "s_pers", "s_np_mae", "drift_abs", "margin_max", "p_send",
                       "p_recv", "mo_lo", "mo_hi", "send_peer", "send_lone", "p_peer", "p_lonef",
                       "cs_val", "cs_sealed", "band_h1", "cov_range")]
print("tables built:", len(TABLES), "| values:", len(V), "| pins:", len(PINS))

tables built: 13 | values: 176 | pins: 23


In [2]:
# Goal: the document as data. PLAN is the full paper in reading order: headings, prose, display
# equations, and the slots where each csv-built table and figure png lands. The builder cell renders
# it without editing a word. Prose is written as f-strings over V, the value dictionary cell 1
# recomputes from the result csvs, so no quantity in this file is typed by hand and a regenerated
# csv rewrites the sentence that describes it. Section word budgets are declared here and checked
# against the built document in the last cell.

# The name is the plain compound of what the system is: federated learning + an agentic
# decision layer + hospital operations. Verified distinct from the existing FedAgent family
# (FedAgent, FedAgent-HPO, FedAgentKE) - no prior FedAgentOps in the literature.
TITLE = ["FedAgentOps: Federated Learning with Agentic Decision Support",
         "for Elective Waiting-List Operations"]
AUTHOR = ["Nilesh Sarkar   |   nileshsarkar.cs@gmail.com",
          "Code, data pipeline and all stored outputs are in the project repository, which retains "
          "its original name: https://github.com/nileshsarkar-ai/fedOPD"]

# heading -> word budget the author declared for that section's prose (captions and tables excluded)
BUDGETS = {"Abstract": 250, "1. Introduction": 600, "2. Literature Review": 400,
           "3. Research Gap": 200, "4. Proposed Methodology": 1000,
           "6. Observations and Findings": 200}

PLAN = [
    ("h1", "Abstract"),
    ("p", f"The elective waiting list in England grew from a monthly mean of {V['list_pre']} "
          f"pathways before March 2020 to {V['list_post']} afterwards, while the share of patients "
          f"waiting beyond 52 weeks rose {V['gt52_fold']}, from {V['gt52_pre']} to {V['gt52_post']}. "
          "Integrated care boards must decide where to expand capacity and when to move demand, yet "
          "the provider-level data behind those decisions cannot be centralised freely. This study "
          "presents FedAgentOps, a systematic evaluation of federated learning, conformal uncertainty "
          f"and frozen allocation rules on {V['n_months']} months of published Referral-to-Treatment "
          f"and KH03 bed data: {V['panel_rows']} provider-specialty-month rows from {V['providers']} "
          "providers. Clients are fitted per provider in level and delta space, federated by "
          "sample-weighted averaging, and extended to gated recurrent units under FedAvg, each "
          f"family trained on {V['full_months']} months of full history and on the "
          f"{V['post_months']} post-Covid months. Forecasts carry split-conformal intervals, two "
          "frozen rule sets convert them into allocation decisions, and the final year was sealed "
          "before any model was fitted and read exactly once. Naive persistence, at MAE "
          f"{V['pers_mae']}, is unbeaten by nine model families at any grain or horizon. "
          f"Level-space FedAvg degrades to {V['lvl_full']}, whereas delta-space FedAvg attains "
          f"{V['dlt_full']} against {V['pool_full']} for centralised pooling. Recurrent clients "
          f"reach {V['a_np_mae']} pooled and {V['a_nf_mae']} federated against {V['a_pers_mae']}, "
          f"RMSE tracking MAE at {V['a_np_rmse']} against {V['a_pers_rmse']}, and alone achieve "
          f"positive R2 on the monthly change, {V['a_np_r2']} and {V['a_nf_r2']}, where every "
          f"linear variant is negative. On the sealed year persistence regains the lead, "
          f"{V['s_pers']} against {V['s_np_mae']}, and every R2 turns negative. The novelty is a "
          "public, fully auditable federated benchmark against operational baselines under a "
          "pre-registered sealed year, identifying delta-space reformulation as what makes "
          "federation viable on non-IID hospitals."),
    ("p", "Keywords: federated learning; NHS elective waiting lists; demand forecasting; conformal "
          "prediction; agentic decision support; healthcare resource allocation."),

    ("h1", "1. Introduction"),
    ("h2", "1.1 Motivation"),
    ("p", "Elective care in England is delivered through Referral-to-Treatment (RTT) pathways, and "
          "every provider reports every treatment function to a national monthly return [16]. The "
          "post-pandemic backlog converted that return into a planning instrument. Across the "
          f"{V['n_months']} months examined here the national list rose from a monthly mean of "
          f"{V['list_pre']} pathways before March 2020 to {V['list_post']} afterwards, and the share "
          f"of patients waiting more than 52 weeks rose {V['gt52_fold']}, from {V['gt52_pre']} to "
          f"{V['gt52_post']}, against a bed capacity base that barely moved over the same period. A "
          "backlog of that magnitude is not managed through a single decision but through a monthly "
          "cycle of decisions taken separately within each integrated care board (ICB). Two "
          "assumptions underpin that cycle. The first is that the near future of each waiting list "
          "is predictable. The second is that the data required to predict it can be pooled into a "
          "single location. Hospital operational data is precisely the setting for which federated "
          "learning was proposed [1], since organisations unable to share records may nonetheless "
          "train a shared model, and healthcare is a canonical motivation for that argument [5]."),
    ("h2", "1.2 Problem Statement"),
    ("p", "Neither assumption survives examination. Hospitals are strongly non-IID clients, because "
          "a large teaching trust and a small specialist centre hold waiting lists differing by "
          "orders of magnitude, and this is exactly the regime in which naive federated averaging is "
          "known to degrade [2]. The forecasting literature independently reports that simple "
          "benchmarks are difficult to beat on noisy operational series [7]. A federated model "
          "unable to beat the rule that next month equals this month has therefore purchased "
          "complexity and privacy machinery for no return. The problem addressed here is not "
          "whether a federated model can be trained on NHS waiting lists, which is straightforward, "
          "but what such a model is worth once it is measured against the baseline an ICB analyst "
          "would otherwise apply, and once it is adjudicated on a year sealed before any model "
          "existed."),
    ("h2", "1.3 Objectives"),
    ("p", "The objectives of this study are as follows. To quantify the accuracy cost of federation "
          "relative to centralised pooling on identical rows, and to establish whether that cost "
          "depends on the space in which the client model is expressed. To measure the margin "
          "available to any trained model over naive forecasting at every spatial grain from "
          "provider to national and at horizons from one to twelve months. To attach distribution-"
          "free uncertainty to those forecasts and to verify its empirical coverage on a year the "
          "calibration never observed. To audit frozen allocation rules driven by the resulting "
          "forecasts, separating decisions that reflect the state of the system from decisions that "
          "are artifacts of the rule's own construction. Each quantity is measured on a held-out "
          "validation year and then adjudicated once on a pre-registered sealed year, so that no "
          "reported figure is the product of a choice made after observing it."),
    ("h2", "1.4 Contributions"),
    ("p", "Audited public panel: a fully reproducible 96-month panel joining RTT waiting lists to "
          "KH03 bed capacity, in which every documented quantity is reproduced by an assertion "
          "inside the notebook that computes it. Baseline closure: persistence is shown to be "
          "unbeaten on MAE by nine model families at every grain and horizon on the validation year, "
          "which places a ceiling on the value of any forecasting layer in this setting. Federation "
          "result: level-space FedAvg is shown to fail on non-IID hospitals, while reformulating the "
          "clients in delta space allows federated linear and recurrent clients to match centralised "
          "pooling. Audited allocation layer: frozen and externally ratified rules whose simulated "
          "decisions separate real populations, together with a structural finding that "
          "redistribution confined within one ICB has almost nowhere to send patients. Prospective "
          "adjudication: a one-shot sealed-year evaluation, pre-registered before execution, that "
          "settles each of the above prospectively rather than retrospectively. The complete "
          "pipeline supporting these contributions is presented in Fig. 1."),

    ("h1", "2. Literature Review"),
    ("p", "Federated averaging was introduced by McMahan et al. as a means of training on "
          "decentralised data without moving it [1]. Zhao et al. documented the failure mode that "
          "matters here, demonstrating that accuracy degrades sharply when client distributions are "
          "non-IID and tracing the effect to weight divergence [2]. FedProx adds a proximal term to "
          "constrain that divergence and remains the standard algorithmic remedy [3]. Kairouz et al. "
          "survey the field and catalogue its open problems, statistical heterogeneity among them "
          "[4]. Within healthcare specifically, Rieke et al. argue that federation is the only "
          "realistic route to multi-institutional models in digital health [5], and Sheller et al. "
          "demonstrated multi-institutional training without sharing patient data on real clinical "
          "imaging [6]. The greater part of this health-federation literature compares learned "
          "models against other learned models, and rarely benchmarks them against the operational "
          "naive forecasts that practitioners already run."),
    ("p", "The forecasting literature supplies precisely that benchmark. The M4 competition, "
          "spanning over 100,000 series and 61 methods, found naive and seasonal-naive references "
          "competitive with considerably more elaborate models on aggregate operational data [7], a "
          "result the standard texts treat as a default expectation rather than a curiosity [8]. The "
          "consequence for federation is material: where the achievable margin over a naive "
          "reference is small, the accuracy cost of distributing training must be measured against a "
          "correspondingly small quantity, and an aggregate improvement of a few percent may vanish "
          "entirely between one year and the next. Applied to hospitals, queueing formulations of "
          "waiting lists date to Worthington [9], while policy analyses of waiting-time initiatives "
          "caution that system dynamics rather than marginal predictive accuracy drive outcomes "
          "[10]. That combination motivates the reporting of calibrated uncertainty in preference to "
          "sharper point forecasts. Conformal prediction supplies distribution-free coverage "
          "guarantees for that purpose under exchangeability alone, with no distributional "
          "assumption imposed on the residuals [11], supported by an accessible modern treatment "
          "[12] and by a regression variant that adapts interval width to local difficulty [13]. The "
          "recurrent clients evaluated here use standard components, namely gated recurrent units "
          "[14], selected over longer-memory architectures because the series are short, trained "
          "with Adam [15]. The gap this body of work leaves open lies at the intersection of its "
          "parts: federated learning evaluated on public health-operations data, against the naive "
          "baselines practitioners actually use, under an evaluation protocol borrowed from clinical "
          "research rather than from machine-learning benchmarking."),

    ("h1", "3. Research Gap"),
    ("p", "The gap this study addresses is the absence of an end-to-end, operationally grounded "
          "evaluation of federated and agentic modelling for improving allocation efficiency in "
          "elective care. Three specific deficiencies follow from the reviewed literature. First, "
          "federated learning for health operations is seldom benchmarked against the naive "
          "forecasts practitioners already use, seldom across multiple algorithms and model "
          "classes under identical conditions, and seldom on public data a reader can download and "
          "re-audit; a model that beats another model may still lose to the arithmetic an analyst "
          "performs by hand. Second, reported gains are rarely tested "
          "prospectively or across training samples of different length: a sealed, pre-registered "
          "final year is standard practice in clinical research and almost absent from federated "
          "forecasting studies, which leaves headline results exposed to retrospective model "
          "selection. Third, allocation rules, increasingly described as agentic, are typically "
          "decoupled from the measured uncertainty of the forecasts that trigger them, and are "
          "rarely audited for decisions that are artifacts of their own construction rather than "
          "evidence about the world. FedAgentOps addresses all three within a single pipeline, "
          "comparing four information regimes and two model classes over two declared training "
          f"samples, {V['full_months']} months of full history and {V['post_months']} post-Covid "
          "months, so that the federation result, the baseline comparison and the allocation layer "
          "are measured on the same rows, under the same temporal cuts, with the same sealed year "
          "adjudicating each."),

    ("h1", "4. Proposed Methodology"),
    ("fig", ("fig_system_architecture.png",
             "Fig. 1. System architecture. Three published sources are harmonised into one "
             "canonical panel, cut at March 2025, and passed through feature engineering into "
             "three modelling layers. The sealed year is unreachable until the final one-shot "
             "evaluation.", 5.6)),
    ("h2", "4.1 Datasets"),
    ("p", "The RTT incomplete-pathway extracts supply one CSV per "
          f"month from April 2018 to March 2026, {V['n_months']} in total, each holding "
          "waiting-list size and band counts per provider and treatment function [16]. "
          "The KH03 return supplies quarterly bed availability and occupancy per trust [17], and "
          "the ODS reference trust identity, sector and ICB membership. Fig. 2 shows the national "
          "series with the Covid break, Fig. 3 the long-wait shares, Fig. 4 the KH03 bed stock and "
          f"occupancy, and Table 1 the era summary. The KH03 series begins in {V['kh_start']}, "
          f"national occupancy spanning {V['occ_lo']} to {V['occ_hi']}, the trough being the first "
          "lockdown quarter. Two training samples are carried through every experiment: "
          f"full history, the {V['full_months']} months ({V['full_years']} years) from April 2018 "
          f"to the development cut, and post-Covid, the {V['post_months']} months "
          f"({V['post_years']} years) from the March 2020 break to the same cut. Every family is "
          "trained on each and scored on identical rows, so the value of the discarded history is "
          "itself measured. Nothing in the pipeline is specific to the NHS: any "
          "hospital system holding a comparable waiting-list and capacity panel can instantiate "
          "the same design."),
    ("fig", ("fig_national_trend.png",
             f"Fig. 2. National RTT waiting list, April 2018 to March 2026 ({V['n_months']} months, "
             f"all {V['providers']} providers and {V['n_tfunc']} treatment functions). The dashed "
             "line is the Covid regime break (March 2020). The shaded band is the sealed year.",
             4.8)),
    ("fig", ("fig_national_longwait_shares.png",
             "Fig. 3. Monthly >18-week and >52-week shares of the national list (denominator: that "
             f"month's total incomplete pathways). The >52-week share rises {V['gt52_fold']} between "
             "era means.", 4.8)),
    ("fig", ("fig_national_beds_kh03.png",
             "Fig. 4. England general and acute overnight beds, open and occupied (left), and the "
             "occupancy rate (right; denominator: beds open that quarter). The dashed line is the "
             f"Covid regime break. The national rate spans {V['occ_lo']} to {V['occ_hi']}. "
             "Occupancy is the third term of the pressure score.", 4.8)),
    ("table", ("era", f"Table 1. National era summary: means of monthly values "
                      f"(eval_national_pre_post.csv; {V['n_pre']} pre-Covid and {V['n_post']} "
                      f"post-Covid months).")),
    ("h2", "4.2 Preprocessing and pipeline"),
    ("p", "The RTT schema breaks in April 2021, when the band columns are renamed and rebased. "
          "Rather than hard-coding column positions, bands are mapped by parsing the week boundaries "
          "out of the column names, so both schemas resolve to the same >18-week and >52-week "
          "counts. The NHS-supplied aggregate row, treatment function C_999, is dropped, since "
          "retaining it doubles every national total. The three sources join on provider code and "
          "month, quarterly KH03 values held forward across their quarter. "
          f"The result is a canonical panel of {V['panel_rows']} "
          f"provider-specialty-month rows and {V['panel_cols']} columns covering {V['providers']} "
          f"providers, of which the {V['public_n']} public trusts hold {V['public_share']} of the "
          "list. The development cut, origins before 2025-04-01, is applied before any target is "
          "constructed, so a sealed-year target cannot be built, and every modelling notebook "
          "carries a sentinel assertion on origin months."),
    ("h2", "4.3 Feature engineering"),
    ("p", "Features are statistics-free: no scaler, encoder or imputer is fitted on any data. Each "
          "provider-specialty series contributes lagged differences at one, "
          "two and three months, the two long-wait shares, general and acute occupancy from KH03, "
          "and referral and completion flows together with their three-month rolling net. Two "
          "arithmetic guards are applied, each documented where it fires. A "
          "provider whose series begins inside the validation year has no prior month, so its "
          "momentum is set to zero rather than dropped. A client with no training rows has no local "
          "model and falls back to persistence. Recurrent inputs are six-month windows of the "
          "log-transformed level, the change divided by 100, and the two shares, all transforms "
          "needing no fitted statistic."),
    ("h2", "4.4 Forecasting layer and uncertainty"),
    ("p", "The baseline is persistence: next month equals this month. Challengers scored on "
          "identical rows are drift, seasonal-naive, gradient-boosted trees over the lag and "
          "occupancy features, and the referral-flow covariates. Uncertainty is a split-conformal "
          "band [11], calibrated on FY 2023-24 residuals, the forecast plus or minus their 90th "
          "percentile:"),
    ("eq", "band(y) = y(t) ± q(0.90) of { |y(t+h) − y(t)| : targets in FY 2023-24 }"),
    ("p", "Calibrating on one year and scoring on the next is the point: coverage measured on the "
          "calibration rows would return 90% by construction. "
          f"Measured on FY 2024-25 instead, coverage is {V['cov_range']} "
          f"against the 90% target, and the band widens from {V['band_h1']} at h=1 to "
          f"{V['band_h12']} at h=12 (Fig. 5)."),
    ("fig", ("fig_c110_conformal_bands_h1.png",
             "Fig. 5. The 90% band around persistence: half-width per horizon (left), calibrated on "
             "FY 2023-24 targets, and its empirical coverage on FY 2024-25 (right), a year the "
             f"calibration never observed. Coverage spans {V['cov_range']}.", 4.8)),
    ("h2", "4.5 Federated models"),
    ("p", "A client is one provider within one specialty. The pool is the "
          f"{V['fl_public']} public providers flagged fl_client, at least 48 months of history. Four information regimes are compared: local-only, centralised pooling, and "
          "federated averaging in two model spaces. Fig. 6 shows what a round moves: each client "
          "fits on its own rows, uploads its parameters and one row count, and receives the average "
          "back. No provider row is transmitted at any point. The level-space client regresses the "
          "next level on the current level and latest change:"),
    ("fig", ("fig_fl_architecture.png",
             "Fig. 6. The federated layer. Clients fit locally and exchange parameters and a row "
             "count with a coordinator that returns the sample-weighted average. Linear clients "
             "require one round, because the average of exact least-squares solutions does not "
             "improve by iterating; recurrent clients run ten.", 5.2)),
    ("eq", "ŷ(t+1) = β0 + β1·y(t) + β2·(y(t) − y(t−1))"),
    ("p", "The delta-space client predicts the change instead and adds it back:"),
    ("eq", "Δŷ(t+1) = a + b·(y(t) − y(t−1)),    ŷ(t+1) = y(t) + Δŷ(t+1)"),
    ("p", "Both are exact least squares per client, so no learning rate or stopping rule exists. "
          "Federation is a single sample-weighted average of the coefficients, in which n(k) "
          "denotes the training row count of client k:"),
    ("eq", "β(fed) = Σ n(k)·β(k) / Σ n(k)"),
    ("p", "Only two or three coefficients and one count ever leave a client. The difference between "
          "the spaces is scale. A level-space intercept encodes the "
          "size of the provider's list, so averaging across providers of widely differing size "
          "produces a model fitting none of them. A delta-space intercept encodes a monthly change, "
          "comparable across providers, so the same average is "
          "meaningful. The recurrent client is a single-layer GRU [14] of input width 4 and hidden "
          "width 32 with a linear head, predicting the scaled change. Federation follows classic "
          "FedAvg [1]: 10 rounds of broadcast, 5 local full-batch Adam [15] steps per round at "
          "learning rate 0.001, then the same weighted average over every weight tensor. The pooled "
          "reference receives the identical budget of 50 epochs. All hyperparameters were declared "
          "before running; there are no sweeps and no early stopping."),
    ("h2", "4.6 Agentic model"),
    ("p", "The agent layer converts forecasts and bands into allocation decisions; Fig. 7 presents "
          "the decision path. Each provider-month receives a pressure score from within-month "
          "percentile ranks of the two long-wait shares and occupancy:"),
    ("eq", "p(i,t) = 0.2·rank(>18w share) + 0.6·rank(>52w share) + 0.2·rank(occupancy)"),
    ("p", "The >52-week share carries the largest weight, being the component that moved "
          f"{V['gt52_fold']} after Covid. Two rule sets consume the score, keyed v1 and v2 and "
          "named for what they decide. The capacity rule set combines pressure "
          "thresholds of 0.70 and 0.40 with the 6-month conformal band, a growth-risk "
          "multiple of 1.20 or a stability multiple of 1.05, and occupancy slack at 0.80, "
          "to trigger EXPAND_CAPACITY. The redistribution rule set classifies senders and receivers "
          "by ICB-internal rank, at or above 0.80 for a sender and at or below 0.30 for a receiver. "
          "One consequence is arithmetic: a receiver rank at or below 0.30 requires an ICB of at "
          "least ceil(1/0.30) = 4 providers, so smaller ICBs cannot produce one. The constants were "
          "ratified and frozen on 2026-07-30, and the freeze notebook re-derives all "
          f"{V['n_dec']} decisions from stored components and asserts bit-for-bit equality."),
    ("fig", ("fig_agent_architecture.png",
             "Fig. 7. The agent layer. Three ranked components form the pressure score; the capacity "
             "rule set also consumes the 6-month conformal band, while redistribution is decided on "
             "ICB-internal ranks alone. Every constant shown was frozen before the decisions were "
             "scored.", 5.2)),
    ("h2", "4.7 Evaluation protocol and metrics"),
    ("p", "FY 2025-26 was sealed before any model was fitted; FY 2024-25 is the validation year, "
          "and one seed, 42, is declared. The two training samples of Section 4.1 are the "
          "only regime boundary applied anywhere. Nothing was adjusted after observing a metric, "
          "thresholds, splits and horizons included. Model quality is scored with three declared "
          "metrics, over n scored rows with level y, forecast ŷ, and monthly changes Δy and Δŷ:"),
    ("eq", "MAE = (1/n) Σ |ŷ(i) − y(i)|"),
    ("eq", "RMSE = √[ (1/n) Σ (ŷ(i) − y(i))² ]"),
    ("eq", "R²(Δ) = 1 − Σ (Δy(i) − Δŷ(i))² / Σ (Δy(i) − mean(Δy))²"),
    ("p", "MAE on the level is primary: it is patients misplaced per provider-month. RMSE exposes "
          "whether a model wins by trimming large errors. R² is computed on the monthly change, "
          "level-R² being dominated by provider size; it is declared secondary, and a negative "
          "value means the model explains less of the movement than the mean monthly change does. "
          "The sealed evaluation is pre-registered in its own notebook header: "
          "full-history regime, linear clients in a val-frozen and a final cut, recurrent clients "
          "final-cut only at seed 42, and test rows whose origin falls in FY 2025-26. It was "
          "executed exactly once."),

    ("h1", "5. Results and Discussion"),
    ("p", "All quantities reported below are read from the stored result CSVs written by the "
          "analysis notebooks and tracked in the repository. Every table in this document is "
          "generated directly from those CSVs by the build notebook, and every sentence quoting a "
          "number is rendered from the same recomputation, so that no figure is transcribed by "
          "hand. The nineteen analysis notebooks (01 to 19) and the cohort-sensitivity audit "
          "(notebook 21) execute headless end to end with zero stored errors."),
    ("h2", "5.1 Persistence is unbeaten at every grain and horizon"),
    ("p", f"On validation, persistence achieves MAE {V['pers_mae']} and RMSE {V['pers_rmse']} over "
          f"{V['n_anchor']} provider-rows at C_110 (Trauma & Orthopaedics), the anchor specialty. No "
          "challenger beats it at any horizon or at any spatial grain. Table 2 reports the full "
          "horizon grid and Fig. 8 presents the same comparison graphically; both are computed on "
          "the 131-provider horizon cohort of notebook 09 rather than on the federated-client rows, "
          f"which is why persistence appears there at h=1 as {V['hz_pers_h1']} on its own eligible "
          "rows. The "
          "result is not an artifact of averaging, since heterogeneity across providers and months "
          f"is substantial. Monthly persistence MAE ranges from {V['mo_lo']} to {V['mo_hi']} across "
          "the validation year (Table 3), and the per-provider distribution is heavily right-skewed "
          f"about a median of {V['med_prov']} (Fig. 9). The target is volatile enough that the "
          "sealed-year outcome reported below is already visible in this dispersion."),
    ("table", ("horizons", "Table 2. Validation MAE by horizon and grain (val_horizons_c110.csv, "
                           "val_horizons_coarse_c110.csv). Provider grain: the 131-provider cohort "
                           "of notebook 09. Challengers are shown for the full-history regime, "
                           "post-only being worse everywhere.")),
    ("table", ("monthly", "Table 3. Persistence MAE per validation origin month, C_110, h=1 "
                          "(eval_c110_persistence_mae_by_month_h1.csv; n = provider rows scored "
                          "that month).")),
    ("fig", ("fig_c110_horizon_mae_by_grain.png",
             "Fig. 8. Persistence against challengers by horizon at provider, ICB and national "
             "grain (validation). The y-scales differ by roughly 30x between grains. Persistence "
             "wins in every panel.", 4.8)),
    ("fig", ("fig_c110_provider_mae_h1.png",
             f"Fig. 9. Per-provider 1-month persistence MAE, C_110, FY 2024-25 "
             f"({V['n_prov_fig']} providers; median {V['med_prov']}).", 4.8)),
    ("h2", "5.2 Delta-space federation matches centralised pooling"),
    ("p", "Table 4 and Fig. 10 report the federation result at the anchor specialty. Level-space "
          f"FedAvg degrades badly on non-IID hospitals, reaching MAE {V['lvl_full']} on full history "
          f"and {V['lvl_post']} post-only against a persistence baseline of {V['pers_mae']}. This is "
          "the weight-divergence failure [2] in its most literal form, and it is precisely what the "
          f"delta reformulation removes. Federated MAE is then {V['dlt_full']} against "
          f"{V['pool_full']} pooled, a gap of {V['fed_pool_gap']} patients. Per client, federated "
          f"coefficients beat local fits at {V['fed_beats_local']} of {V['n_clients']} clients, with "
          f"median per-client MAE of {V['med_local']} local, {V['med_fed']} federated and "
          f"{V['med_naive']} naive. Table 5 scales the comparison to every specialty in the pool, "
          f"{V['n_val_all']} validation rows, where the federated-minus-pooled gap has median "
          f"{V['fp_med']} MAE, worst {V['fp_worst']} and best {V['fp_best']}. A delta model edges "
          f"persistence at {V['dlt_wins']} scored specialties, the largest margin arising at "
          f"{V['dlt_best_code']}, {V['dlt_best_name']}, at {V['dlt_best_mae']} against "
          f"{V['dlt_best_pers']}. The baseline closure of the previous subsection is therefore "
          "specific to the anchor. The margins involved remain small in both directions, the best "
          f"win being {V['dlt_best_win']} MAE and the worst loss {V['dlt_worst_loss']} MAE."),
    ("p", "Fig. 11 places all four information regimes against the baseline in every specialty "
          "simultaneously, which separates the variants more sharply than the anchor alone. "
          f"Delta-space FedAvg beats persistence at {V['dlt_wins']} specialties with a median gap of "
          f"{V['dlt_medgap']} MAE, centralised pooling at {V['pool_wins']} with {V['pool_medgap']}, "
          f"local-only at {V['local_wins']} with {V['local_medgap']}, and the local-intercept "
          f"variant at {V['dlta_wins']} with {V['dlta_medgap']}. Federating the slope while "
          "retaining a local intercept therefore costs more than federating both, because the "
          "intercept is where the sample-weighted average earns its keep: a single client's own "
          f"history estimates its mean monthly change poorly. {V['any_wins']} specialties have at "
          "least one variant below persistence, and they are the same specialties the delta family "
          "wins."),
    ("table", ("fl", f"Table 4. Validation MAE / RMSE at C_110 on the identical {V['n_anchor']} rows "
                     f"(fl_summary_c110.csv).")),
    ("fig", ("fig_c110_fl_models_h1.png",
             "Fig. 10. Level-space federation fails; delta-space federation matches centralised "
             f"pooling (validation, both regimes, the same {V['n_anchor']} rows at C_110).", 4.8)),
    ("fig", ("fig_fl_algorithms_by_specialty_h1.png",
             "Fig. 11. Every linear information regime against the naive baseline, per specialty, "
             "full-history training (below zero favours the trained model). Level-space FedAvg is "
             "off this scale by design and is shown at the anchor in Fig. 10.", 4.8)),
    ("table", ("scaleup", f"Table 5. FL scale-up: full-history validation MAE per specialty "
                          f"(fl_scaleup_by_specialty.csv; {V['n_tfunc']} specialties, "
                          f"{V['fl_public']} public fl_client providers, {V['n_val_all']} validation "
                          f"rows). Local and federated models are delta-space. Bold = best trained "
                          f"model where it also beats persistence. X01 has zero validation rows in "
                          f"this pool, leaving {V['n_spec']} scored specialties.")),
    ("h2", "5.3 Recurrent clients: parity overall, a margin at the anchor"),
    ("p", "Table 6 compares the linear and recurrent families per specialty on identical "
          "window-eligible rows. Fig. 12 places the four trained families against persistence at "
          "the anchor, Fig. 13 gives the comparison in all four combinations of architecture and "
          "regime, Fig. 14 shows the federated pair as the neural notebook itself plotted it, and "
          "Fig. 15 lays the same federated gap out specialty by specialty. The result depends "
          "entirely on which architecture is compared. "
          f"Federated, the GRU beats linear at {V['nvl_fedavg_full']} specialties in the full regime "
          f"and {V['nvl_fedavg_post']} post-Covid, with median gaps of "
          f"{V['nvl_fedavg_full_med']} and {V['nvl_fedavg_post_med']} MAE, which constitutes parity. "
          f"Pooled, the same GRU beats linear at {V['nvl_pooled_full']} and "
          f"{V['nvl_pooled_post']}, with median gaps of {V['nvl_pooled_full_med']} and "
          f"{V['nvl_pooled_post_med']} MAE, which constitutes a real if modest advantage. The "
          "nonlinearity is therefore worth something, and federated averaging is what removes it, "
          "since averaging nonlinear weight tensors loses more than averaging two coefficients. The "
          f"same effect is visible within a single specialty at {V['ill_code']}, {V['ill_name']}, "
          f"where the pooled GRU scores {V['ill_np']} against {V['ill_nf']} federated, with linear "
          f"FedAvg at {V['ill_lf']}. The anchor is the exception. The pooled GRU reaches "
          f"{V['a_np_mae']} and the federated GRU {V['a_nf_mae']} against persistence "
          f"{V['a_pers_mae']} on {V['n_anchor_nn']} identical rows. Every sub-persistence score "
          "recorded at the anchor in either regime is recurrent, since no linear variant falls below "
          f"{V['a_pers_mae']}, and a declared three-seed check confirms the finding (Table 7) with "
          "all six full-regime scores below persistence. Dataset size does not decide the winner: "
          f"the size-gap correlation is r = {V['r_size']} (Fig. 16), and the four largest relative "
          f"recurrent wins are the administrative X-codes ({V['nn_best_codes']}), which are the "
          "specialties holding no pre-Covid history. The signal being modelled remains thin "
          f"throughout, the highest delta-space R2 recorded anywhere being {V['r2_max']}."),
    ("table", ("neural", "Table 6. Recurrent against linear clients: full-regime validation MAE per "
                         "specialty on identical window-eligible rows (fl_neural_summary.csv). "
                         "Bold = best trained model where it also beats persistence.")),
    ("table", ("multiseed", f"Table 7. Declared multi-seed check of the C_110 recurrent result, "
                            f"n={V['n_anchor_nn']}, persistence {V['a_pers_mae']} "
                            f"(fl_neural_c110_multiseed.csv). Verdict rule fixed before running: "
                            f"confirmed iff all three full-regime seeds beat persistence.")),
    ("fig", ("fig_c110_fl_linear_vs_neural_h1.png",
             "Fig. 12. Linear against recurrent families at C_110 on identical eligible rows "
             f"(validation, n={V['n_anchor_nn']}).", 4.8)),
    ("fig", ("fig_neural_vs_linear_4panel_h1.png",
             "Fig. 13. Neural against linear MAE, one point per specialty, for both architectures "
             "and both regimes, on identical validation rows per point. Points below the diagonal "
             f"favour the neural model. Pooled neural wins {V['nvl_pooled_full']} and "
             f"{V['nvl_pooled_post']}; federated neural wins {V['nvl_fedavg_full']} and "
             f"{V['nvl_fedavg_post']}.", 5.4)),
    ("fig", ("fl_neural_vs_linear.png",
             "Fig. 14. The federated pair of Fig. 13 as written by the neural notebook itself, from "
             "the same stored scores. Both panels carry linear FedAvg MAE on the horizontal axis "
             "and neural FedAvg MAE on the vertical; the shared vertical label is printed once, on "
             "the left. Points below the diagonal favour the neural model.", 5.0)),
    ("fig", ("fig_fl_neural_gap_by_specialty_h1.png",
             "Fig. 15. Neural minus linear FedAvg MAE per specialty, both regimes (below zero "
             f"favours neural): {V['nvl_fedavg_full']} below zero in the full regime and "
             f"{V['nvl_fedavg_post']} post-Covid. Parity, not a takeover.", 4.8)),
    ("fig", ("fig_fl_neural_gap_vs_size_h1.png",
             f"Fig. 16. Relative neural-linear FedAvg gap against specialty size (Pearson r = "
             f"{V['r_size']}, {V['n_spec_nn']} specialties). Size does not decide the winner. The "
             f"largest neural wins are the X-codes with no pre-Covid history ({V['nn_best_codes']}).",
             4.8)),
    ("h2", "5.4 Beyond MAE: error shape and explained variance"),
    ("p", "MAE alone cannot establish whether a model wins by trimming large errors or whether it "
          "explains any of the month-to-month movement, so Table 8 reports the full metric set at "
          f"the anchor on the {V['n_anchor_nn']} window-eligible validation rows. Two conclusions "
          "follow. First, RMSE moves with MAE and never reverses a verdict: the pooled GRU is best "
          f"on both, at {V['a_np_mae']} and {V['a_np_rmse']} against persistence at "
          f"{V['a_pers_mae']} and {V['a_pers_rmse']}, and the RMSE-to-MAE ratio sits near "
          f"{V['rmse_mae_ratio']} for every model. No variant wins by reshaping the error "
          "distribution; all are making the same kind of mistake, in the same proportion, slightly "
          "more or less often. Second, the R2 on deltas is the sharpest discriminator in the study. "
          "It is declared secondary because level-R2 is meaningless in this setting, provider size "
          "dominating it, but on the change it poses exactly the right question: does the model "
          "explain any of the month-to-month movement beyond its mean? Only the recurrent clients "
          f"do, at {V['a_np_r2']} pooled and {V['a_nf_r2']} federated. Every linear variant is "
          f"negative, from {V['a_lp_r2']} pooled to {V['a_lf_r2']} federated, as is persistence at "
          f"{V['a_pers_r2']}. The linear family is not merely failing to beat the baseline; it is "
          "failing to explain any movement at all."),
    ("table", ("metrics", f"Table 8. The full metric set at C_110 on the identical "
                          f"{V['n_anchor_nn']} window-eligible validation rows, full-history regime "
                          f"(fl_neural_summary.csv). R2 is computed on the monthly change and is "
                          f"declared secondary; a negative value means the model explains less of "
                          f"the movement than the mean monthly change does.")),
    ("p", "The sealed year removes even that. Table 11 carries RMSE and R2 alongside MAE for the "
          "one-shot, and every model there is negative on R2, the pooled GRU included at "
          f"{V['s_np_r2']}. The one genuine signal found anywhere in the project, an ability to "
          "explain roughly 1.3% of the variance in monthly change, did not survive into a "
          f"prospective year. Sealed RMSE at the anchor is far lower, {V['s_rmse_anchor']} against "
          f"{V['a_pers_rmse']} on validation, but that is a property of the year rather than of the "
          "models, since FY 2025-26 simply contained fewer extreme monthly swings at C_110. The "
          "effect does not generalise: averaged across all scored specialties, persistence RMSE "
          f"rises from {V['v_rmse_mean']} on validation to {V['s_rmse_mean']} on the sealed year. "
          "Each year is scored on its own rows, so these comparisons describe the target rather than "
          "model skill."),
    ("h2", "5.5 Discarding pre-Covid history costs accuracy"),
    ("p", "Once models are scale-free, discarding pre-Covid history is costly. Full-history training "
          f"beats post-only at {V['dlt_reg_clin']} clinical specialties for linear FedAvg and at "
          f"{V['neu_reg_clin']} for the federated GRU, each family compared within its own fixed "
          f"validation rows (Fig. 17). The {V['n_xcodes']} X-codes sit exactly at zero, having no "
          f"pre-Covid rows to drop. The largest penalty falls at {V['dlt_reg_worst']}, "
          f"Gynaecology, at {V['dlt_reg_worstv']} MAE for the linear family and "
          f"{V['neu_reg_worstv']} for the recurrent one."),
    ("p", "The penalty is a property of the data rather than of federation. Fig. 18 repeats the "
          "comparison for all four linear information regimes, and full history wins in every one: "
          f"{V['local_reg_wins']} specialties for local-only at a median {V['local_reg_med']} MAE, "
          f"{V['pool_reg_wins']} for pooled at {V['pool_reg_med']}, {V['dlt_reg_wins']} for "
          f"delta-space FedAvg at {V['dlt_reg_med']}, and {V['dlta_reg_wins']} for the "
          f"local-intercept variant at {V['dlta_reg_med']}. {V['dlt_reg_worst']} is the worst case "
          "in three of the four. A model that never leaves its own provider is hurt most by the "
          "shorter history, which is what would be expected if the pre-Covid rows supply sample size "
          "rather than a different regime. The regime break that destroys level-space models barely "
          "matters in delta space, which is a second piece of evidence that the problem confronting "
          "level models is scale rather than distribution shift."),
    ("fig", ("fig_fl_covid_regime_effect_h1.png",
             "Fig. 17. Cost of training on post-Covid data only (post minus full MAE), per "
             "specialty, for federated linear and federated neural clients. Each family is compared "
             "within its own csv on its own fixed validation rows: linear from "
             "fl_scaleup_by_specialty.csv, neural from fl_neural_summary.csv.", 4.8)),
    ("fig", ("fig_fl_regime_effect_4panel_h1.png",
             "Fig. 18. The same regime comparison for all four linear information regimes (above "
             "zero means dropping pre-Covid history hurts). Specialty order is shared across "
             "panels, so a row can be read straight across.", 5.6)),
    ("h2", "5.6 Allocation rules and the geography of receivers"),
    ("p", f"Table 9 summarises the frozen rules over {V['n_dec']} provider-months ({V['dec_from']} "
          f"to {V['dec_to']}, {V['dec_providers']} providers), Table 10 reports the three-window "
          "stress test, and Fig. 19 shows the redistribution classes against what happened six "
          f"months later. The frozen redistribution rule flags {V['n_send']} sender months and "
          f"{V['n_recv']} receiver months. Their pressure separation, median {V['p_send']} against "
          f"{V['p_recv']}, is a sanity check rather than evidence, because the rule classifies on "
          "pressure rank and therefore cannot fail it. The evidence lies in the outcomes. Sender "
          "flags precede the largest realised 6-month deteriorations, a median of "
          f"{V['send_change']} patients overall and {V['send_change_last']} in the final window, "
          "although the ordering reverses in the first stress window, where STATUS_QUO months "
          f"deteriorated more, at {V['sq_change_first']} (Table 10)."),
    ("p", f"An audit decomposition splits the {V['n_send']} sender flags in two (Fig. 20). "
          f"{V['send_peer']}, with median pressure {V['p_peer']}. The other {V['send_lone']} "
          "ICB-months, in which a within-ICB percentile rank is 1.0 by definition, so that the flag "
          "is mechanical and the pressure behind it is ordinary: median "
          f"{V['p_lonef']}, against a cohort median of {V['p_cohort']}. The structural finding is "
          f"geographic. Only {V['has_recv']} of the {V['n_send']} sender months have any in-ICB "
          f"receiver that month, and {V['small_icb_n']} of {V['n_send']}, or {V['small_icb']}, "
          "occur in ICBs with fewer than four cohort providers, where the frozen threshold renders a "
          "receiver arithmetically impossible. The capacity rule set fails its own checks in both "
          "directions: its band-conditioned receiver clause never fires, zero times in "
          f"{V['n_dec']} provider-months, and its EXPAND_CAPACITY months preceded a median realised "
          f"change of {V['exp_change']} against {V['sq_change']} under STATUS_QUO (Table 9). That "
          "trigger selects small stable lists in place of growing ones."),
    ("table", ("agent", f"Table 9. Frozen agent rules: decision classes over {V['n_dec']} "
                        f"provider-months (agent_frozen_summary.csv; csv rule keys: v1 = capacity, "
                        f"v2 = redistribution).")),
    ("table", ("stress", f"Table 10. Redistribution-rule stress test: three disjoint 6-month origin "
                         f"windows partitioning all {V['n_dec']} decisions "
                         f"(eval_c110_agent_stress_h6.csv).")),
    ("fig", ("fig_c110_agent_v2_h6.png",
             "Fig. 19. Redistribution decision classes (left) and the median realised 6-month "
             f"change that followed each class (right), {V['dec_from']} to {V['dec_to']}.", 4.8)),
    ("fig", ("fig_c110_agent_sender_split_h6.png",
             f"Fig. 20. The {V['n_send']} sender flags split by whether the provider had any peer in "
             "its ICB that month (left), and the pressure each group carries (right; boxes are "
             "quartiles, whiskers the 1.5 IQR range, fliers omitted). The lone-ICB flags sit near "
             f"the cohort median of {V['p_cohort']}, so they carry no pressure signal.", 4.8)),
    ("h2", "5.7 The prospective year"),
    ("p", "Table 11 and Fig. 21 examine the anchor, and Table 12 reports the complete one-shot with "
          "each model's validation MAE joined beside its sealed score. At C_110 persistence takes "
          f"the lead back, {V['s_pers']} against {V['s_np_mae']} for the pooled GRU "
          f"(n={V['n_sealed']}), so the three-seed-confirmed validation edge did not replicate "
          "prospectively. That margin is not decisive. Under the prospective cohort rule of Table 13 "
          f"and Fig. 22, persistence alone moves from {V['cs_sealed_stored']} to {V['cs_sealed']}, "
          f"which exceeds its {V['s_anchor_gap']} gap to the pooled GRU, so the anchor verdict is a "
          "tie inside cohort sensitivity before the seed spread is even counted. The broader pattern "
          f"went the other way and strengthened. The pooled GRU beats persistence at "
          f"{V['s_np_wins']} specialties, the federated GRU and final-cut linear FedAvg at "
          f"{V['s_nf_wins']} and {V['s_lf_wins']}, and the exact validated linear fits at "
          f"{V['s_vf_wins']}. The largest win arises at {V['s_best_code']}, {V['s_best_name']}, at "
          f"{V['s_best_mae']} against {V['s_best_pers']} (Fig. 23). What dominates all of this is "
          "the year itself. Per-specialty persistence MAE drifted by a median of "
          f"{V['drift_med']} from validation to sealed year, the median {V['drift_abs']}, with a "
          f"range of {V['drift_lo']} to {V['drift_hi']} (Fig. 24), while the largest "
          f"model-over-baseline margin measured anywhere in the project {V['margin_max']}."),
    ("table", ("sealed_anchor", f"Table 11. Sealed-year FY 2025-26 at C_110 (n={V['n_sealed']}): "
                                f"MAE, RMSE and R2 on the monthly change, with each model's "
                                f"validation-year MAE beside it (sealed_year_eval.csv).")),
    ("table", ("sealed_full", "Table 12. Sealed-year FY 2025-26: the complete one-shot MAE per "
                              "specialty (sealed_year_eval.csv). Linear and neural models trained "
                              "on the final cut (targets through March 2025); the last column is "
                              "the exact validated linear fit. Bold = best trained model where it "
                              "also beats persistence.")),
    ("table", ("cohort", "Table 13. Cohort-rule sensitivity at C_110: the stored eligibility rule "
                         "(reporting in March 2026, inside the sealed year) against the strictly "
                         "prospective rule (history before the March 2025 cut), full-history "
                         "regime (cohort_sensitivity.csv). The one-shot in Table 12 stands verbatim "
                         "as the pre-registered record; this sensitivity post-dates the unsealing. "
                         "Model ordering is identical under both rules.")),
    ("fig", ("fig_c110_sealed_one_shot_h1.png",
             f"Fig. 21. The one-shot verdict at the anchor: persistence retakes the lead by "
             f"{V['s_anchor_gap']} MAE. Each year is scored on its own rows (validation "
             f"n={V['n_anchor_nn']}, sealed n={V['n_sealed']}).", 4.8)),
    ("fig", ("fig_c110_cohort_sensitivity_h1.png",
             "Fig. 22. Validation MAE at C_110 under the stored eligibility rule and under the "
             "strictly prospective one. Every model moves by roughly the same amount, so the "
             "ordering on which the conclusions rest does not depend on the defective rule. Each "
             "rule is scored on its own rows.", 4.8)),
    ("fig", ("fig_sealed_gap_by_specialty_h1.png",
             "Fig. 23. Sealed-year MAE gap to persistence per specialty, three final-cut models "
             f"(below zero favours the model): {V['s_nf_wins']} to {V['s_np_wins']} specialties "
             f"below zero.", 4.8)),
    ("fig", ("fig_sealed_persistence_drift_h1.png",
             "Fig. 24. Persistence MAE change from validation to sealed year, per specialty "
             f"(median {V['drift_med']}). The year moves more than the models.", 4.8)),
    ("h2", "5.8 Critical analysis"),
    ("p", "Four weaknesses deserve to be stated plainly. The first is the cohort rule. An "
          "independent audit established that fl_client eligibility requires a provider to be "
          "reporting in the panel's final month, March 2026, which falls inside the sealed year. "
          "Cohort membership therefore used information unknowable at the development cut. This is "
          "selection look-ahead rather than target leakage, since no future value enters any model, "
          f"and the sensitivity in Table 13 quantifies it: the validation anchor moves to "
          f"{V['cs_val']} and the sealed anchor to {V['cs_sealed']}, with model ordering unchanged. "
          f"It nonetheless means the {V['s_anchor_gap']} MAE sealed verdict at the anchor should be "
          "read as a tie. The second is the redistribution rule's design sequence. It was written "
          f"after inspecting the capacity rules' outcome check on the same {V['n_dec']} "
          "provider-months, so its outcome separation is in-sample, and a prospective test on "
          "sealed-year origins remains available and undeclared. The third is publication lag: NHS "
          "England releases each extract roughly six to eight weeks after month end, so every model "
          "reported here, persistence included, is compared on an information set a deployed system "
          "would not possess. A real forecast starts from the latest published month and therefore "
          "operates near the h=3 error level of Table 2. The fourth is scope. One seed drives the "
          f"recurrent results, cuDNN training is not bit-reproducible, and {V['n_scores']} "
          "model-regime-specialty scores were computed, so isolated sub-persistence results are "
          "expected by chance alone. The three-seed check exists precisely because a single seed is "
          "not evidence."),
    ("h2", "5.9 Use cases"),
    ("p", "Four deployment shapes follow from these results, and together they describe how the "
          "federated and agentic layers combine into an operational-efficiency instrument rather "
          "than a forecasting exercise. The first is a shared forecasting service across an ICB or "
          f"a provider collaborative. Because delta-space federation costs {V['fed_pool_gap']} MAE "
          "against pooling, participants may contribute two coefficients and a row count in place "
          "of patient-level extracts, which removes the data-sharing agreement from the critical "
          "path without altering what the model is able to do. The second is capacity planning "
          "under an explicit error budget. The deployable output is not a point forecast but the "
          "interval: at one month ahead, planners should treat a provider's list as known to "
          f"roughly {V['band_h1']}, and any decision that flips inside that width is not supported "
          "by the data. The third is triage support for redistribution. The sender rule reliably "
          "marks the providers that go on to deteriorate most, so it functions as an early-warning "
          "watchlist that concentrates scarce management attention where deterioration is coming, "
          "although the geography result means it cannot be operated as an automatic matcher inside "
          "ICB boundaries. The fourth is portability. The pipeline consumes only a waiting-list "
          "panel, a capacity return and an organisational reference, none of them specific to the "
          "NHS, so any hospital chain or regional health system holding comparable data can "
          "implement the same stack end to end: federate without moving records, attach calibrated "
          "intervals, and drive frozen, auditable allocation rules from them."),
    ("h2", "5.10 Implications"),
    ("p", "Four practical conclusions follow. First, federation is close to free. In delta space it "
          "costs essentially nothing against centralised pooling (Tables 4 and 5), so a data-sharing "
          "agreement is not a prerequisite for a shared waiting-list model, and the privacy argument "
          "for federation may be made on its own merits rather than traded against accuracy. "
          "Second, the operational value resides in the interval and the rules, not in the point "
          "forecast. The anchor's edge over persistence held across three seeds (Table 7) and still "
          "did not survive a prospective year whose own difficulty moved by tens of percent (Table "
          f"12); calibrated conformal bands, at {V['cov_range']} coverage, are the honest planning "
          "input instead, and the frozen rules convert them into decisions a committee can audit "
          "line by line. Third, allocation efficiency is bounded by structure before statistics. "
          f"The frozen rules separate populations cleanly (Tables 9 and 10), but {V['small_icb']} "
          "of sender flags occur where no local receiver can exist, so widening the receiving pool "
          "across ICB boundaries or into the independent sector would improve system throughput "
          "more than any refinement of the score. Fourth, the evaluation template itself is an "
          "implication: a sealed year, declared seeds and machine-checked numbers cost little to "
          "adopt and would make reported gains in this literature considerably more trustworthy."),

    ("h1", "6. Observations and Findings"),
    ("p", "Six findings summarise the study. One, the naive baseline is strong: persistence at MAE "
          f"{V['pers_mae']} over {V['n_anchor']} rows was not beaten by nine model families at any "
          "grain from provider to national, or at any horizon from one to twelve months. Two, model "
          f"space decides whether federation works at all: level-space FedAvg reached "
          f"{V['lvl_full']}, worse than doing nothing, while delta-space FedAvg reached "
          f"{V['dlt_full']} against {V['pool_full']} for centralised pooling, a difference of "
          f"{V['fed_pool_gap']} patients on the same rows. Three, capacity is not the binding "
          "constraint: recurrent clients reproduced the linear result almost exactly in aggregate, "
          f"winning {V['nn_pair_share']} of {V['nn_pairs']} client-specialty pairs, and the highest "
          f"delta-space R2 recorded anywhere was {V['r2_max']}, so what limits accuracy is signal "
          "rather than model class. Four, validation edges are fragile: the anchor's "
          f"sub-persistence recurrent scores, {V['a_np_mae']} and {V['a_nf_mae']} against "
          f"{V['a_pers_mae']}, survived three seeds and still did not replicate on the sealed year, "
          f"where persistence led at {V['s_pers']} against {V['s_np_mae']}. Five, the year moves "
          f"more than the models: the median {V['drift_abs']} in per-specialty persistence error "
          "between the two years, while the largest model margin measured anywhere "
          f"{V['margin_max']}. Six, the allocation layer fails structurally rather than "
          f"statistically: of {V['n_send']} sender flags, {V['send_peer']} and {V['send_lone']} "
          f"ICB-months where the rank is 1.0 by construction, and {V['small_icb']} of all sender "
          "months sit in ICBs too small to contain a receiver."),

    ("h1", "7. Conclusion and Future Scope"),
    ("p", "On fully public data, under a pre-registered sealed-year protocol, FedAgentOps establishes "
          "two results. Federated linear and recurrent clients match centralised training at no "
          "accuracy cost once the models predict changes rather than levels. And none of them "
          "reliably beats naive persistence by more than a rounding error at any grain or horizon. "
          "The deployable system is therefore persistence together with conformal uncertainty, "
          "frozen and transparent allocation rules, and federation as the privacy mechanism rather "
          "than as an accuracy mechanism; because the pipeline consumes only published returns, the "
          "same design transfers to any health system holding comparable panels. Four directions "
          f"remain open. The client pool can be widened to the {V['fl_indep']} independent-sector "
          "fl_client providers, whose RTT series arrive without KH03 capacity. The agent layer "
          "requires cross-ICB and independent-sector receiving rules, together with a prospective "
          "test of the redistribution family on sealed-year origins. Richer exogenous signals, "
          "among them referral pipelines and workforce, may supply the signal the current features "
          "lack. And federated algorithms beyond one-shot averaging, FedProx [3] among them, remain "
          "untested here, although the delta-space result suggests the binding constraint is the "
          "signal rather than the algorithm."),

    ("h1", "References"),
]

REFS = [
    "[1] B. McMahan, E. Moore, D. Ramage, S. Hampson, and B. A. y Arcas, "
    "“Communication-efficient learning of deep networks from decentralized data,” in "
    "Proc. AISTATS, 2017, pp. 1273-1282.",
    "[2] Y. Zhao, M. Li, L. Lai, N. Suda, D. Civin, and V. Chandra, “Federated learning "
    "with non-IID data,” arXiv:1806.00582, 2018.",
    "[3] T. Li, A. K. Sahu, M. Zaheer, M. Sanjabi, A. Talwalkar, and V. Smith, “Federated "
    "optimization in heterogeneous networks,” in Proc. MLSys, 2020.",
    "[4] P. Kairouz et al., “Advances and open problems in federated learning,” "
    "Foundations and Trends in Machine Learning, vol. 14, no. 1-2, 2021.",
    "[5] N. Rieke et al., “The future of digital health with federated learning,” "
    "npj Digital Medicine, vol. 3, no. 119, 2020.",
    "[6] M. J. Sheller et al., “Federated learning in medicine: facilitating "
    "multi-institutional collaborations without sharing patient data,” Scientific "
    "Reports, vol. 10, no. 12598, 2020.",
    "[7] S. Makridakis, E. Spiliotis, and V. Assimakopoulos, “The M4 competition: 100,000 "
    "time series and 61 forecasting methods,” International Journal of Forecasting, "
    "vol. 36, no. 1, pp. 54-74, 2020.",
    "[8] R. J. Hyndman and G. Athanasopoulos, Forecasting: Principles and Practice, 3rd ed. "
    "Melbourne: OTexts, 2021.",
    "[9] D. J. Worthington, “Queueing models for hospital waiting lists,” Journal of "
    "the Operational Research Society, vol. 38, no. 5, pp. 413-422, 1987.",
    "[10] S. A. Kreindler, “Policy strategies to reduce waits for elective care: a "
    "synthesis of international evidence,” British Medical Bulletin, vol. 95, no. 1, "
    "pp. 7-32, 2010.",
    "[11] V. Vovk, A. Gammerman, and G. Shafer, Algorithmic Learning in a Random World. "
    "New York: Springer, 2005.",
    "[12] A. N. Angelopoulos and S. Bates, “Conformal prediction: a gentle "
    "introduction,” Foundations and Trends in Machine Learning, vol. 16, no. 4, "
    "pp. 494-591, 2023.",
    "[13] Y. Romano, E. Patterson, and E. Candès, “Conformalized quantile "
    "regression,” in Proc. NeurIPS, 2019.",
    "[14] K. Cho et al., “Learning phrase representations using RNN encoder-decoder for "
    "statistical machine translation,” in Proc. EMNLP, 2014.",
    "[15] D. P. Kingma and J. Ba, “Adam: a method for stochastic optimization,” in "
    "Proc. ICLR, 2015.",
    "[16] NHS England, “Consultant-led Referral to Treatment waiting times data,” "
    "https://www.england.nhs.uk/statistics/statistical-work-areas/rtt-waiting-times/, "
    "accessed 2026.",
    "[17] NHS England, “Bed availability and occupancy data (overnight, KH03),” "
    "https://www.england.nhs.uk/statistics/statistical-work-areas/"
    "bed-availability-and-occupancy/, accessed 2026.",
]
PLAN += [("ref", r) for r in REFS]
print("plan blocks:", len(PLAN), "| budgeted sections:", len(BUDGETS))

plan blocks: 131 | budgeted sections: 6


In [3]:
# Goal: render the plan into docs/paper/fedagentops_paper.docx. Single column, Times New Roman,
# 12 pt body per the declared format, smaller tables with the bold marks, figures centred
# at their declared widths.

from docx import Document                        # python-docx: writes .docx files directly
from docx.shared import Pt, Inches, RGBColor     # point sizes, image widths, font colours
from docx.enum.text import WD_ALIGN_PARAGRAPH    # paragraph alignment constants
from docx.enum.table import WD_TABLE_ALIGNMENT   # centres the table block on the page
from docx.oxml.ns import qn                      # w: namespace lookups for raw OOXML
from docx.oxml import OxmlElement                # header-cell shading has no python-docx API

doc = Document()
section = doc.sections[0]
for side in ("top_margin", "bottom_margin", "left_margin", "right_margin"):
    setattr(section, side, Inches(0.8))   # the default 1-inch margins cost the 12-page budget
normal = doc.styles["Normal"]
normal.font.name = "Times New Roman"
normal.font.size = Pt(12)
normal.paragraph_format.space_after = Pt(3)
for name, size in (("Heading 1", 14), ("Heading 2", 12.5)):
    style = doc.styles[name]
    style.font.name = "Times New Roman"
    style.font.size = Pt(size)
    style.font.color.rgb = RGBColor(0, 0, 0)   # the default template renders headings blue
    style.paragraph_format.space_before = Pt(6)   # the template's 12 pt costs the page budget
    style.paragraph_format.space_after = Pt(3)

CENTER = WD_ALIGN_PARAGRAPH.CENTER
LEFT, RIGHT = WD_ALIGN_PARAGRAPH.LEFT, WD_ALIGN_PARAGRAPH.RIGHT


def para(text, size=12, bold=False, align=None, space_after=6):
    # one paragraph with a single run, so font size and weight apply to the whole text
    p = doc.add_paragraph()
    run = p.add_run(text)
    run.font.size = Pt(size)
    run.font.bold = bold
    if align is not None:
        p.alignment = align
    p.paragraph_format.space_after = Pt(space_after)


def shade(cell, fill):
    # python-docx exposes no shading API, so the w:shd element is written onto the cell directly
    shd = OxmlElement("w:shd")
    shd.set(qn("w:val"), "clear")
    shd.set(qn("w:fill"), fill)
    cell._tc.get_or_add_tcPr().append(shd)


def fill_cell(cell, text, bold, size, align):
    # table cells come with one empty paragraph; write into it with zero spacing
    p = cell.paragraphs[0]
    p.paragraph_format.space_after = Pt(0)
    p.paragraph_format.space_before = Pt(0)
    p.alignment = align
    run = p.add_run(str(text))
    run.font.size = Pt(size)
    run.font.bold = bold


def render_table(disp, bold, caption):
    para(caption, size=9, space_after=3)
    t = doc.add_table(rows=len(disp) + 1, cols=len(disp.columns))
    t.style = "Table Grid"
    t.alignment = WD_TABLE_ALIGNMENT.CENTER
    # wide per-specialty tables need the smaller step; the short summary tables can breathe
    size = 9.5 if len(disp.columns) <= 6 else 8.5
    for j, col in enumerate(disp.columns):
        fill_cell(t.rows[0].cells[j], col, bold=True, size=size, align=CENTER)
        shade(t.rows[0].cells[j], "D9D9D9")   # grey header, so the head reads as a head in print
    for i, (key, row) in enumerate(disp.iterrows(), start=1):
        for j, col in enumerate(disp.columns):
            marked = bold is not None and col in bold.columns and bool(bold.loc[key, col])
            # first column is a label, the rest are numbers and read better right-aligned
            fill_cell(t.rows[i].cells[j], row[col], bold=marked, size=size,
                      align=LEFT if j == 0 else RIGHT)
    para("", size=6, space_after=0)   # a hair of air between a table and the next block


def render_fig(png, caption, width):
    doc.add_picture(str(FIGS / png), width=Inches(width))
    doc.paragraphs[-1].alignment = CENTER   # add_picture leaves the image in a fresh paragraph
    para(caption, size=8.5, align=CENTER)


for line in TITLE:
    para(line, size=16, bold=True, align=CENTER, space_after=2)
for line in AUTHOR:
    para(line, size=10.5, align=CENTER, space_after=2)
for kind, payload in PLAN:
    if kind == "h1":
        doc.add_heading(payload, 1)
    elif kind == "h2":
        doc.add_heading(payload, 2)
    elif kind == "p":
        para(payload)
    elif kind == "eq":
        para(payload, size=11, align=CENTER, space_after=6)   # display equation on its own line
    elif kind == "ref":
        para(payload, size=8.5, space_after=1)
    elif kind == "table":
        key, caption = payload
        render_table(*TABLES[key], caption)
    elif kind == "fig":
        render_fig(*payload)

OUT = ROOT / "docs" / "paper" / "fedagentops_paper.docx"
doc.save(OUT)
print(f"written {OUT.resolve()}  ({OUT.stat().st_size:,} bytes)")

written C:\Users\heave\Documents\NHS UK FL Agentic AI\docs\paper\fedagentops_paper.docx  (1,582,627 bytes)


In [4]:
# Goal: verify the stored docx against the csvs and against the declared section budgets. Reopens
# the saved file so every check runs on what is actually on disk, not on the in-memory document.

from docx import Document

saved = Document(OUT)
paras = [p.text for p in saved.paragraphs]
text = "\n".join(paras)                        # prose + headings + captions + references


def wordcount(s):
    return len([w for w in s.split() if any(ch.isalnum() for ch in w)])


# walk the rendered paragraphs, attributing prose to whichever h1 heading last opened
section, prose = None, {}
heads = {payload for kind, payload in PLAN if kind == "h1"}
bodies = {payload for kind, payload in PLAN if kind == "p"}
for p in paras:
    if p in heads:
        section = p
    elif section is not None and p in bodies:
        prose[section] = prose.get(section, 0) + wordcount(p)

print("section word counts (budget in brackets):")
for sec, n in prose.items():
    budget = BUDGETS.get(sec)
    print(f"  {sec:<32} {n:>5}" + (f"  [{budget}]  {n / budget - 1:+.0%}" if budget else ""))

for sec, budget in BUDGETS.items():
    # a section that drifts more than 15% off its declared length is a rewrite, not a rounding error
    assert abs(prose[sec] / budget - 1) <= 0.15, (sec, prose[sec], budget)

assert len(saved.tables) == 13, len(saved.tables)             # trips if a table dropped out
assert len(saved.inline_shapes) == 24, len(saved.inline_shapes)   # trips if a png went missing
for pin in PINS:
    assert pin in text, pin   # trips if a result csv changed after the prose was written

print(f"\nfedagentops_paper.docx verified: {wordcount(text):,} words of text, "
      f"{sum(prose.values()):,} of them prose in {len(prose)} sections, "
      f"{len(saved.tables)} tables, {len(saved.inline_shapes)} figures, "
      f"all {len(PINS)} headline pins found")

section word counts (budget in brackets):
  Abstract                           286  [250]  +14%
  1. Introduction                    628  [600]  +5%
  2. Literature Review               392  [400]  -2%
  3. Research Gap                    228  [200]  +14%
  4. Proposed Methodology           1149  [1000]  +15%
  5. Results and Discussion         2531
  6. Observations and Findings       228  [200]  +14%
  7. Conclusion and Future Scope     186

fedagentops_paper.docx verified: 7,377 words of text, 5,628 of them prose in 8 sections, 13 tables, 24 figures, all 23 headline pins found
